In [0]:
%run ../functions/functions

In [0]:
# Nome do banco de dados onde a tabela será criada ou utilizada
database_name = "fato"

# Nome da tabela de destino para importações municipais
table_name = "ft_importacoes_mun"

# Caminho completo no formato <database>.<table> para operações Spark SQL
target_path = f"{database_name}.{table_name}"

# Nome da chave primária utilizada para identificar unicamente os registros
pk = "SK_IMPORTACAO_MUN"

In [0]:
# Caminho do diretório Silver no Data Lake para a tabela consolidada de importações municipais.
# O caminho utiliza o protocolo abfss para acessar o container 'silver' no storage account 'stgbbb'.
# O diretório 'balancacomercial/IMPORTACAO_MUN_CONSOLIDADA/' armazena os dados consolidados de importação municipal.
silver_path_s = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/IMPORTACAO_MUN_CONSOLIDADA/"

In [0]:
# Lê os dados do caminho Delta Lake especificado em 'silver_path_s' e carrega em um DataFrame Spark.
# Este DataFrame será utilizado para análises e transformações subsequentes.
df = spark.read.format("delta").load(silver_path_s)

In [0]:
# Cria ou substitui uma temporary view chamada "df_imp_mun" a partir do DataFrame df.
# Essa view pode ser utilizada em consultas SQL temporárias durante a sessão Spark atual.
df.createOrReplaceTempView("df_imp_mun")

In [0]:
# Consulta para selecionar e transformar dados de importações municipais
# - ano_mes: concatenação do ano e mês da importação
# - SH4: código SH4 do produto
# - CO_PAIS: código do país de origem
# - CO_MUN: código do município
# - KG_LIQUIDO: peso líquido da importação
# - VL_FOB: valor FOB da importação
# Fonte: view temporária 'df_imp_mun'

query = """
select
  concat(e.CO_ANO,'-',e.CO_MES) as ano_mes,
  e.SH4,
  CO_PAIS,
  CO_MUN,
  KG_LIQUIDO,
  VL_FOB
from df_imp_mun  e
 """

In [0]:
# Executa a query SQL definida anteriormente e armazena o resultado em um DataFrame Spark.
# Esta consulta seleciona e transforma colunas específicas da tabela temporária 'df_imp_mun'.
df = spark.sql(query)

In [0]:
# Salva o DataFrame 'df' como uma tabela Hive no caminho especificado por 'target_path',
# utilizando 'pk' como chave primária. Função customizada definida em ../functions/functions.
save_hive_table(df, target_path, pk)